# Fit $E_J$ and $E_C$ from target transmon frequencies

This notebook is a Python translation of `get_EjEc.ipynb`.
It inlines the transmon Hamiltonian construction that was previously imported from the Julia module.

In [1]:
import numpy as np
from scipy.linalg import eigvalsh
from scipy.optimize import minimize

In [2]:
def build_transmon_ops(n_charge_cut):

    dim_t = 2 * n_charge_cut + 1

    charge_range = np.arange(-n_charge_cut, n_charge_cut + 1, dtype=float)



    n_charge = np.diag(charge_range)



    cos_phi = 0.5 * (

        np.diag(np.ones(dim_t - 1), 1) + np.diag(np.ones(dim_t - 1), -1)

    ) - np.eye(dim_t)



    return n_charge, cos_phi





def build_transmon_hamiltonian(Ec, Ej, ng, n_charge_cut):

    n_charge, cos_phi = build_transmon_ops(n_charge_cut)

    charge_values = np.diag(n_charge)

    charging_term = np.diag(4 * Ec * (charge_values - ng) ** 2)

    hamiltonian = charging_term - Ej * cos_phi

    return hamiltonian, n_charge, cos_phi





def fit_transmon_params(f01_target, alpha_target, ng_target, n_charge_cut=20):

    """Find Ej and Ec that reproduce the target f01 and anharmonicity at a given ng."""

    Ec_guess = -alpha_target

    Ej_guess = (f01_target - alpha_target) ** 2 / (8 * Ec_guess)

    initial_guess = np.array([Ej_guess, Ec_guess], dtype=float)



    def loss(params):

        Ej_try, Ec_try = params



        if not np.isfinite(Ej_try) or not np.isfinite(Ec_try) or Ej_try <= 0 or Ec_try <= 0:

            return np.inf



        hamiltonian, _, _ = build_transmon_hamiltonian(Ec_try, Ej_try, ng_target, n_charge_cut)

        evals = eigvalsh(hamiltonian)



        f01_calc = evals[1] - evals[0]

        alpha_calc = (evals[2] - evals[1]) - f01_calc

        return (f01_calc - f01_target) ** 2 + (alpha_calc - alpha_target) ** 2



    result = minimize(loss, initial_guess, method="Nelder-Mead")

    Ej_opt, Ec_opt = result.x

    return Ej_opt, Ec_opt





def get_ej_ec_band(f01_target, alpha_target, n_charge_cut=20):

    """Calculate Ej and Ec bounds between ng = 0 and ng = 0.5."""

    Ej_0, Ec_0 = fit_transmon_params(f01_target, alpha_target, 0.0, n_charge_cut=n_charge_cut)

    Ej_05, Ec_05 = fit_transmon_params(f01_target, alpha_target, 0.5, n_charge_cut=n_charge_cut)



    Ej_band = {"Ej_lower": float(min(Ej_0, Ej_05)), "Ej_upper": float(max(Ej_0, Ej_05))}

    Ec_band = {"Ec_lower": float(min(Ec_0, Ec_05)), "Ec_upper": float(max(Ec_0, Ec_05))}

    return Ej_band, Ec_band


In [4]:
Ej_band, Ec_band = get_ej_ec_band(3.078582882, -0.2398)

print(f"Ej lies between: {Ej_band['Ej_lower']} and {Ej_band['Ej_upper']}")
print(f"Ec lies between: {Ec_band['Ec_lower']} and {Ec_band['Ec_upper']}")

Ej lies between: 6.721693408002593 and 6.804665232747694
Ec lies between: 0.19909370795685966 and 0.2019816554169556
